In [40]:
from langchain_openai import ChatOpenAI
from structed_relations_generate import Relation_Generate
from pydantic import BaseModel, Field
from typing import List, Optional
from rebuild_graph_transformer import GraphTransformer
from langchain_core.documents import Document
from langchain_neo4j import Neo4jGraph
from process_pdf import extract_blocks_from_pdf
import time
from langchain_community.graphs.graph_document import GraphDocument

In [34]:
# prompt_strings
sys_relation_summary_str = """
You are a computer system knowledge graph architect. Please strictly follow these rules to summarize relationship types:
- Core Rules
  1. Relationship types must be concise **verbs** only; no other components.
  2. Ignore modifiers.
  3. Relationships must be **verbs** that can connect two conceptual entities to form a coherent sentence.

"""
"""
    您是计算机网络知识图谱架构师，请严格遵循以下规则归纳关系类型：
    【核心规则】  
    1. 关系类是简洁的动词，不包含其他成分。    
    2. 忽略修饰词。
    3. 关系必须为**动词**，且能够连接两个概念实体，构成合理的句子。
"""
sys_extract_relation_str = """
        你是计算机系统领域的知识图谱关系抽取专家。请按照下列规则从给定文本中抽取关系：
        1. 你需要总结关系三元组，其中head和tail是总结性的概念实体，不是指某个具体的事物。
        2. head_id 和 tail_id 字段指的是核心实体的名称，不要有修饰词，且必须是名词。
        3. **relation字段中绝对不能出现宾语**，让 head_id + relation + tail_id 构成一个合理的句子，其中head_id 以及 tail_id 是名词，relation 是谓语。
        4. 所有 专有名词，必须放在 head_id 或 tail_id 字段中，**绝对不能**出现在 relation 字段中。
        5. 对并列的宾语生成多个三元组，即关注表示并列的关键字符如：“和”，“顿号”，“以及”，“或” 等 eg: (head:A, relation:xxx, tail:B 和 C) 拆分为 (head:A, relation:xxx, tail:B) 和 (head:A, relation:xxx, tail:C)
        7. 注意关系的存在条件/状语应当放在relation_condition字段中。
        8. 强调head和tail是总结性的，不是具体的某个事物。用英文输出
"""   


"""
6. 可添加“释义”关系以及概念解释类型节点用于概念解释。
Role: You are an expert in relation extraction for knowledge graphs, specializing in the computer networking domain.
Task: From the provided text, extract relational triples (head, relation, tail) by strictly adhering to the following rules:
Conceptual Abstraction: The head and tail must be summarized conceptual entities, representing a general class or idea, not a specific instance.
Entity Naming (head_id, tail_id):Entities must be core concepts named as nouns.Do not include any modifiers (e.g., adjectives, quantifiers like "some," "many").
Relation Formatting (relation):The relation must be a concise predicate verb,Crucially, the relation field must NOT contain an object.The final structure head_id + relation + tail_id must form a simple, coherent sentence (e.g., "Firewall filters Traffic").
Placement of Proper Nouns: All proper nouns and technical terms must be placed in the head_id or tail_id fields. They must never appear in the relation field.
Handling Conjunctions: If an entity has a relationship with multiple other entities listed in a series (using "and," "or," commas, etc.), create a separate triple for each one.
Example: "A connects to B and C" must be extracted as two triples:
(head: A, relation: connects_to, tail: B)
(head: A, relation: connects_to, tail: C)
Definitions: For conceptual explanations, you may use a specific "defines" relation, linking a term to a "Concept Explanation" node.
Conditional Information: If a relationship only exists under a specific condition or is achieved through a certain method, capture this information in a separate relation_condition field. Do not include it in the relation itself.

你是计算机网络领域的知识图谱关系抽取专家。请按照下列规则从给定文本中抽取关系：
1. 尽可能忽略 **不重要的， 与计算机网络无关的** 三元组。
2. head_id 和 tail_id 字段指的是核心实体的名称，尽量不要有修饰词，且必须是名词，不抽取简单的修饰词，如“很多”，“一些”，“大部分”等。
3. 强调，**relation字段中绝对不能出现宾语**，让 head_id + relation + tail_id 构成一个合理的句子，其中head_id 以及 tail_id 是名词，relation 是谓语。
4. 对于所以专业技术/协议名词，必须放在 head_id 或 tail_id 字段中，**绝对不能**出现在 relation 字段中，同时若有中文翻译，（英文+中文）形式。
5. 对并列的宾语生成多个三元组，即关注表示并列的关键字符如：“和”，“顿号”，“以及”，“或” 等 eg: (head:A, relation:xxx, tail:B 和 C) 拆分为 (head:A, relation:xxx, tail:B) 和 (head:A, relation:xxx, tail:C)
6. 可添加“释义”关系以及概念解释类型节点用于概念解释。
7. 注意关系的存在条件/状语，如“在某条件下”、“通过某方式”等，可放在relation_condition字段中。 
"""

"""
1. 理解文本整体含义，再总结关系三元组；
2. 只抽取计算机网络相关的核心概念实体间的关系。
3. 关系必须为**动词**，关系中禁止出现宾语及具体事物名词，注意关系的状语/存在条件。
让head_id + relation_id + tail_id 构成一句语法完整，语义通顺的句子。    
2. **语法驱动**：分析句子主谓宾及依存结构；并列句要拆分成多条独立关系。
6. **节点可细化**：节点名称可带必要修饰定语以区分同名实体（例如“边缘路由器”、“核心交换机”），且有识别性，但是描述中切勿包含谓语。

要注意分析一个句子的语法关系，主语、谓语、宾语等成分，对于并列结构的句子，要拆分成多个关系进行抽取
请先理解语义，根据语义进行关系抽取，不要只看表面词语
忽略简单的修饰词，如“很多”，“一些”，“大部分”等
忽略简单无意义或不重要的关系

可以添加“释义/解释”这一关系
rel加入“存在条件”属性，rel中绝对不能出现宾语，最好是抽象的关系描述，即不要出现具体的设备名称、协议名称等，不要“是”，“不是”，重中之重
对于rel中无宾语这一点，若直接输出无法避免，加入监测llm，让监测llm对其拆分//删除，同时也可用于拆分并列宾语

rel中也可区分type和id
node.id中按情况，加入修饰定语，最后embedding找出不同定语/无定语的相似节点，让llm再次进行判断（根据存在的重要性，意义deng)，有些进行合并，其余的可以转化为层次结构（节点属性中也加入上下文信息，便于llm区分）
统计node的度，一定意义上反映节点的重要性
"""



"""
你是计算机网络领域的知识图谱抽取器，负责从给定文本中基于语义准确地抽取关系三元组（头部实体 — 关系 — 尾部实体）。严格遵循以下规则与约束，只按规则抽取，不要输出额外示例或格式说明。

核心任务与原则：

1. **基于语义与句法**：必须先分析句子的语法成分（主语、谓语、宾语、并列结构、从句等）和语义角色后再抽取关系；不要只看表面关键词或简单字符串匹配。
2. **并列拆分**：遇到并列结构（主语并列、宾语并列、谓语并列、并列从句）时，应拆分为多条独立关系分别抽取，确保不合并不同逻辑的行为。
3. **关系（rel）要求**：

   * 关系必须为**抽象的动词短语**（尽量使用动词或动宾短语的动词部分），描述实体之间的互动或功能；**不得包含宾语成分**；不得出现具体设备、协议或客体名称；不得使用“是/不是”等判断句式。
   * 关系应简洁（中文不超过6字），避免数值、时间、单位或具体实现细节。
   * 关系应尽量通用化与规范化（例如“使用TCP协议”应抽为“使用”；“建立链路”可抽为“建立/连接”视语义而定）。
4. **节点（实体）要求**：

   * 头部与尾部应为具体的名词短语或名词性表述（可以带必要的修饰定语以区分概念，如“核心路由器”“边缘交换机”“应用层服务”），但不要把协议或设备名并入关系部分。
   * 当语义需要区分细类时，可在节点名中加入修饰定语；节点应尽可能保留能指向原文的语义信息以便消歧。
5. **忽略无意义/琐碎信息**：

   * 忽略简单修饰词（例如“很多”“一些”“大部分”等）对关系的影响，不把它们作为关系或节点重要部分。
   * 忽略明显无意义或不重要的关系（如“存在于句子中但不表行为或功能的描述”）。
6. **避免过度具体**：不要将关系具体化为某个协议版本、具体端口、精确数值或实现细节；如需表现这些信息，应保留在实体或证据中，而不是关系本身。
7. **全面性与谨慎性并重**：尽量从一段文本中抽取**尽可能多**的、语义明确的关系，但对每条关系必须能在原文中找到支持的语义证据；不要凭推测生成无根关系。
8. **去冗与规范化**：对于多次重复表达或只是表述方式不同但语义相同的关系，应在抽取时标准化为单一抽象关系（例如“建立连接”“建立链路”“连接上”统一为“连接”）。
9. **禁止项**：关系字段**绝对不能**包含宾语或具体客体名称；关系中不得出现具体设备名、协议名、端口号、数值、单位、或“是/不是”等判断词。
10. **证据意识**：每条抽取的三元组必须能在原文中被定位（即存在支撑该关系的语句或子句）。
11. **语境敏感**：若同一短语在不同上下文表示不同语义（例如“支持”可表示“协议支持功能”或“设备支持某功能”），应根据上下文区分并抽为不同节点或关系，避免模糊合并。
12. **输出要求**：只进行抽取行为，不添加解释、示例或额外注释；确保每条关系符合上述所有规则。
请依以上规则从给定文本中尽可能全面且严谨地抽取所有符合条件的关系三元组。

"""

'\n你是计算机网络领域的知识图谱抽取器，负责从给定文本中基于语义准确地抽取关系三元组（头部实体 — 关系 — 尾部实体）。严格遵循以下规则与约束，只按规则抽取，不要输出额外示例或格式说明。\n\n核心任务与原则：\n\n1. **基于语义与句法**：必须先分析句子的语法成分（主语、谓语、宾语、并列结构、从句等）和语义角色后再抽取关系；不要只看表面关键词或简单字符串匹配。\n2. **并列拆分**：遇到并列结构（主语并列、宾语并列、谓语并列、并列从句）时，应拆分为多条独立关系分别抽取，确保不合并不同逻辑的行为。\n3. **关系（rel）要求**：\n\n   * 关系必须为**抽象的动词短语**（尽量使用动词或动宾短语的动词部分），描述实体之间的互动或功能；**不得包含宾语成分**；不得出现具体设备、协议或客体名称；不得使用“是/不是”等判断句式。\n   * 关系应简洁（中文不超过6字），避免数值、时间、单位或具体实现细节。\n   * 关系应尽量通用化与规范化（例如“使用TCP协议”应抽为“使用”；“建立链路”可抽为“建立/连接”视语义而定）。\n4. **节点（实体）要求**：\n\n   * 头部与尾部应为具体的名词短语或名词性表述（可以带必要的修饰定语以区分概念，如“核心路由器”“边缘交换机”“应用层服务”），但不要把协议或设备名并入关系部分。\n   * 当语义需要区分细类时，可在节点名中加入修饰定语；节点应尽可能保留能指向原文的语义信息以便消歧。\n5. **忽略无意义/琐碎信息**：\n\n   * 忽略简单修饰词（例如“很多”“一些”“大部分”等）对关系的影响，不把它们作为关系或节点重要部分。\n   * 忽略明显无意义或不重要的关系（如“存在于句子中但不表行为或功能的描述”）。\n6. **避免过度具体**：不要将关系具体化为某个协议版本、具体端口、精确数值或实现细节；如需表现这些信息，应保留在实体或证据中，而不是关系本身。\n7. **全面性与谨慎性并重**：尽量从一段文本中抽取**尽可能多**的、语义明确的关系，但对每条关系必须能在原文中找到支持的语义证据；不要凭推测生成无根关系。\n8. **去冗与规范化**：对于多次重复表达或只是表述方式不同但语义相同的关系，应在抽取时标准化为单一抽象关系（例如“建立连接”“建立链路”“连接上”统一为“

In [4]:
def load_texts_from_wiki(source: list[str]) -> list[str]:
    text = []
    for src in source:
        with open(f"./{src}", "r", encoding="utf-8") as file:
            lines = [line.strip() for line in file]
        text.append(lines[3])
    return text

In [5]:
def load_texts_from_pdf(source: list[str]) -> list[str]:
    text = []
    for src in source:
        blocks = extract_blocks_from_pdf(pdf_file_path=src, block_length=2000, title_font_size=15, end_title="References")
        text.extend(blocks) # extend instead of append
    return text
    

In [6]:

def summarize_relations(texts: list[str], relationship_generator: Relation_Generate) -> None:
    res_relationships:List[str]= []
    for text in texts:
        time.sleep(5)
        relationships = relationship_generator.generate_relations(relations=res_relationships, inputs=text)
        res_relationships.extend(relationships)
    return res_relationships

In [7]:

class ComputerSystemRelation(BaseModel): # 计算机系统关系, 对应知识图谱中的边
    head: str = Field(
        ...,
        description=(
           "关系主语：表示该关系的主体实体，可以取自上下文的名词短语。",
        )
    )
    head_type: str = Field(
        ...,
        description=(
            "主语实体类型：表示该实体所属的类别/类型",
        )
    )
    relation_type: str = Field(
        ...,
        description=(
            "关系类型：表示主语实体与宾语实体之间的关系，取自预定义的关系类型列表。"
        )
    )
    relation_condition: str = Field(
        ...,
        description=(
            "关系状语：表示该关系存在的前提/时间、空间条件/方式，可以取自上下文的状语。"
            " 'all' 表示无条件成立"
        )
    )
    tail: str = Field(
        ...,
        description=(
           "关系宾语：表示该关系的客体实体，可以取自上下文的名词短语。",
        )
    )
    tail_type: str = Field(
        ...,
        description=(
            "尾部所属的实体类型：表示该实体所属的类别/类型",
        )
    )
    source_document: str = Field(
        ...,
        description="抽取该关系的 源语句/段落 的回显"
    )


class without_function_calling_graphDocument(BaseModel):
    relationships: List[ComputerSystemRelation] = Field(
        ...,
        description="提取出的关系列表",
    )

In [33]:
def generate_graph_doc(transformer: GraphTransformer, text:list[str], source) -> list:
    docs = []
    if len(source) == 1 :
        for txt in text:
            doc = Document(page_content=txt, metadata={'source': source})
            docs.append(doc)
    else:
        for i, txt in enumerate(text):
            doc = Document(page_content=txt, metadata={'source': source[i]})
            docs.append(doc)

    graph_doc = []
    for doc in docs:
        time.sleep(10)  # 避免请求过快
        print("处理文档，来源：", doc.metadata['source'])
        graph = transformer.extraction_from_document(doc)["gpd_list"]
        graph_doc.extend(graph)
    return graph_doc

In [12]:
llm_qwen = ChatOpenAI(
        model="qwen-plus-2025-09-11",
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        api_key="sk-aec12d19a21049a2a56fa32acb855dae",
    )

res_nodes = []
res_relationships = []
text1 = "wiki_computer_network_txt/wiki_computer_network/40874.txt"
text2 = "wiki_computer_network_txt/wiki_computer_network/40614.txt"
text3 = "zhwiki_computer_network_txt/computer_network_xml/5201.txt"
text4 = "zhwiki_computer_network_txt/computer_network_xml/5249.txt"
# source = [text1, text2]
source = ["Kernel_(operating_system).pdf"]
print("开始加载pdf文本...")
text = load_texts_from_pdf(source)

开始加载pdf文本...


In [ ]:
# text = load_texts_from_wiki(source)

print("pdf文本加载完成，开始生成关系类型...")
relationship_generator = Relation_Generate(
        llm=llm_qwen,
        node_labels=res_nodes,
        function_calling=False,
        extract_prompt_str=sys_relation_summary_str
        )

res_relationships = summarize_relations(texts=text, relationship_generator=relationship_generator)
print("最终关系类型：", res_relationships)
with open("./data/res_relationships.pkl", "wb") as f:
    import pickle
    pickle.dump(res_relationships, f)






开始加载pdf文本...
pdf文本加载完成，开始生成关系类型...
1
2
3


2025-11-26 21:09:33,765 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'relations': ['controls', 'prevents', 'mitigates', 'facilitates', 'manages', 'arbitrates', 'optimizes', 'handles', 'translates', 'protects', 'invokes', 'allocates']}
1
2
3


2025-11-26 21:09:39,435 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'relations': ['allocates', 'manages', 'protects', 'facilitates', 'arbitrates', 'translates', 'handles']}
1
2
3


...

{'relations': ['separates', 'implements', 'reliesOn', 'distinguishes', 'combines', 'delegates', 'runsIn', 'communicatesThrough']}
1
2
3


2025-11-26 21:10:10,498 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'relations': ['builds', 'virtualizes', 'supports', 'extends', 'replaces', 'combines', 'influences', 'runs', 'provides']}
1
2
3


2025-11-26 21:10:16,039 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'relations': []}
最终关系类型： ['controls', 'prevents', 'mitigates', 'facilitates', 'manages', 'arbitrates', 'optimizes', 'handles', 'translates', 'protects', 'invokes', 'allocates', 'allocates', 'manages', 'protects', 'facilitates', 'arbitrates', 'translates', 'handles', 'requests', 'invokes', 'provides', 'uses', 'switches', 'accesses', 'delegates', 'enforces', 'implements', 'simulates', 'isolates', 'verifies', 'restricts', 'separates', 'implements', 'reliesOn', 'distinguishes', 'combines', 'delegates', 'runsIn', 'communicatesThrough', 'builds', 'virtualizes', 'supports', 'extends', 'replaces', 'combines', 'influences', 'runs', 'provides']


In [35]:
with open("./data/res_relationships.pkl", "rb") as f:
    import pickle
    res_relationships = pickle.load(f)

transformer = GraphTransformer(
        llm=llm_qwen,
        allowed_nodes=res_nodes,
        allowed_relationships=res_relationships,
        function_calling=False,
        pydantic_object=without_function_calling_graphDocument,
        function_calling_pydantic=None,
        prompt_str=sys_extract_relation_str
)

graph_doc = generate_graph_doc(transformer=transformer, text=text, source=source)
with open("./data/graph_doc_raw.pkl", "wb") as f:
    import pickle
    pickle.dump(graph_doc, f)



# url = "neo4j://localhost:7688"
# username = "neo4j"
# password = "wiki_pass"
# graph = Neo4jGraph(
#                     url=url,
#                     username=username,
#                     password=password
#                 )
# graph.query("MATCH (n) DETACH DELETE n")
# graph.add_graph_documents(graph_doc)
# print("图数据已成功添加到 Neo4j 数据库中。")

处理文档，来源： ['Kernel_(operating_system).pdf']


2025-11-27 20:51:32,500 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'relationships': [{'head': 'Kernel', 'head_type': 'ComputerSystemComponent', 'relation_type': 'controls', 'relation_condition': 'all', 'tail': 'everything in the system', 'tail_type': 'SystemResource', 'source_document': "A kernel is a computer program at the core of a computer's operating system that always has complete control over everything in the system."}, {'head': 'Kernel', 'head_type': 'ComputerSystemComponent', 'relation_type': 'prevents', 'relation_condition': 'all', 'tail': 'conflicts between different processes', 'tail_type': 'ProcessConflict', 'source_document': 'The kernel is also responsible for preventing and mitigating conflicts between different processes.'}, {'head': 'Kernel', 'head_type': 'ComputerSystemComponent', 'relation_type': 'mitigates', 'relation_condition': 'all', 'tail': 'conflicts between different processes', 'tail_type': 'ProcessConflict', 'source_document': 'The kernel is also responsible for preventing and mitigating conflicts between different proce

2025-11-27 20:51:50,903 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'relationships': [{'head': 'RAM', 'head_type': 'Memory', 'relation_type': 'stores', 'relation_condition': 'all', 'tail': 'program instructions and data', 'tail_type': 'Data', 'source_document': 'Random-access memory (RAM) is used to store both program instructions and data.'}, {'head': 'Kernel', 'head_type': 'Component', 'relation_type': 'controls', 'relation_condition': 'when insufficient memory is available', 'tail': 'memory access for processes', 'tail_type': 'Resource', 'source_document': 'The kernel is responsible for deciding which memory each process can use, and determining what to do when insufficient memory is available.'}, {'head': 'Kernel', 'head_type': 'Component', 'relation_type': 'provides', 'relation_condition': 'all', 'tail': 'convenient methods for applications to use I/O devices', 'tail_type': 'Interface', 'source_document': 'The kernel provides convenient methods for applications to use these devices which are typically abstracted by the kernel so that applications

...

{'relationships': [{'head': 'Microkernel', 'head_type': 'ComputerSystemComponent', 'relation_type': 'facilitates', 'relation_condition': 'all', 'tail': 'Separation of mechanism and policy', 'tail_type': 'DesignPrinciple', 'source_document': 'Because the mechanism and policy are separated, the policy can be easily changed to, e.g., require the use of a security token. In a minimal microkernel just some very basic policies are included, and its mechanisms allow what is running on top of the kernel (the remaining part of the operating system and the other applications) to decide which policies to adopt'}, {'head': 'Monolithic kernel', 'head_type': 'ComputerSystemComponent', 'relation_type': 'combines', 'relation_condition': 'all', 'tail': 'Mechanism and policy', 'tail_type': 'DesignPrinciple', 'source_document': "the 'privileged mode' architectural approach melds together the protection mechanism with the security policies, while the major alternative architectural approach, capability-ba

2025-11-27 20:53:36,878 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'relationships': [{'head': 'Operating system kernel', 'head_type': 'Component', 'relation_type': 'facilitates', 'relation_condition': 'all', 'tail': 'Hardware abstraction', 'tail_type': 'Function', 'source_document': 'Programs can be directly loaded and executed on the "bare metal" machine, provided that the authors of those programs are willing to work without any hardware abstraction or operating system support.'}, {'head': 'Ancillary programs', 'head_type': 'Component', 'relation_type': 'formed', 'relation_condition': 'as these were developed', 'tail': 'Basis of early operating system kernels', 'tail_type': 'Concept', 'source_document': 'Eventually, small ancillary programs such as program loaders and debuggers were left in memory between runs, or loaded from ROM. As these were developed, they formed the basis of what became early operating system kernels.'}, {'head': 'RC 4000 Multiprogramming System', 'head_type': 'System', 'relation_type': 'introduced', 'relation_condition': 'in 

2025-11-27 20:53:53,002 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'relationships': [{'head': 'Virtual addressing', 'head_type': 'Concept', 'relation_type': 'is achieved through', 'relation_condition': 'most commonly', 'tail': 'memory management unit', 'tail_type': 'Component', 'source_document': 'Virtual addressing is most commonly achieved through a built-in memory management unit.'}, {'head': 'Highest privilege level', 'head_type': 'Concept', 'relation_type': 'has', 'relation_condition': 'throughout different architectures', 'tail': 'supervisor mode', 'tail_type': 'Mode', 'source_document': 'The highest privilege level has various names throughout different architectures, such as supervisor mode, kernel mode, CPL0, DPL0, ring 0, etc.'}, {'head': 'Highest privilege level', 'head_type': 'Concept', 'relation_type': 'has', 'relation_condition': 'throughout different architectures', 'tail': 'kernel mode', 'tail_type': 'Mode', 'source_document': 'The highest privilege level has various names throughout different architectures, such as supervisor mode, k

In [42]:
with open("./data/graph_doc_raw.pkl", "rb") as f:
    import pickle
    graph_doc = pickle.load(f)
print(len(graph_doc))
# 统计总节点和关系数
total_nodes = 0
total_relationships = 0
for gpd in graph_doc:
    total_nodes += len(gpd.nodes)
    total_relationships += len(gpd.relationships)

print(f"Total nodes: {total_nodes}")
print(f"Total relationships: {total_relationships}")

8
Total nodes: 201
Total relationships: 163
